In [ ]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.nn import global_mean_pool, GATConv
from sklearn.metrics import r2_score
import numpy as np
import pickle
import pandas as pd
import random
from torch.cuda.amp import GradScaler, autocast

def set_seed(seed=42):
    """Set all random generators used by the distillation workflow."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

class GATTeacher(nn.Module):
    """Teacher GAT used to provide hidden features during distillation."""
    def __init__(self, node_dim, edge_dim, global_dim, hidden_dims, dropout=0.2, gat_heads=4):
        super().__init__()
        if not hidden_dims:
            raise ValueError("hidden_dims must not be empty")
        if any(h_dim % gat_heads != 0 for h_dim in hidden_dims):
            raise ValueError("Each hidden dimension must be divisible by gat_heads")

        self.gat_heads = gat_heads
        self.node_norm = nn.BatchNorm1d(node_dim)
        self.edge_norm = nn.BatchNorm1d(edge_dim) if edge_dim else None
        self.global_norm = nn.BatchNorm1d(global_dim) if global_dim else None
        self.global_mlp = None
        if global_dim:
            self.global_mlp = nn.Sequential(
                nn.Linear(global_dim, hidden_dims[-1]),
                nn.ReLU(),
                nn.Dropout(dropout)
            )

        self.convs = nn.ModuleList()
        self.conv_norms = nn.ModuleList()
        in_dim = node_dim
        for h_dim in hidden_dims:
            self.convs.append(GATConv(
                in_channels=in_dim,
                out_channels=h_dim // gat_heads,
                heads=gat_heads,
                concat=True,
                dropout=dropout,
                edge_dim=edge_dim if edge_dim else None,
                add_self_loops=True
            ))
            self.conv_norms.append(nn.LayerNorm(h_dim))
            in_dim = h_dim

        self.dropout = nn.Dropout(dropout)
        self.final_dim = hidden_dims[-1] * (2 if global_dim else 1)
        self.output_mlp = nn.Sequential(
            nn.Linear(self.final_dim, self.final_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(self.final_dim // 2, 1)
        )

    def forward(self, data, return_feat=False):
        x = self.node_norm(data.x)
        edge_attr = getattr(data, 'edge_attr', None)
        if self.edge_norm is not None and edge_attr is not None:
            edge_attr = self.edge_norm(edge_attr)

        u = getattr(data, 'u', None)
        if self.global_norm is not None and u is not None:
            u = self.global_norm(u)

        for conv, conv_norm in zip(self.convs, self.conv_norms):
            x = conv(x, data.edge_index, edge_attr=edge_attr)
            x = conv_norm(x)
            x = F.elu(x)
            x = self.dropout(x)

        node_pool = global_mean_pool(x, data.batch)
        h = torch.cat([node_pool, self.global_mlp(u)], dim=1) if u is not None else node_pool
        out = self.output_mlp(h).squeeze(-1)
        return (out, h) if return_feat else out

class EnhancedGAT(nn.Module):
    """Student GAT trained with hard labels, soft labels, and feature hints."""
    def __init__(self, node_dim, edge_dim, global_dim, hidden_dims, dropout=0.2, gat_heads=4):
        super().__init__()
        if not hidden_dims:
            raise ValueError("hidden_dims must not be empty")
        if any(h_dim % gat_heads != 0 for h_dim in hidden_dims):
            raise ValueError("Each hidden dimension must be divisible by gat_heads")

        self.gat_heads = gat_heads
        self.node_norm = nn.BatchNorm1d(node_dim)
        self.edge_norm = nn.BatchNorm1d(edge_dim) if edge_dim else None
        self.global_norm = nn.BatchNorm1d(global_dim) if global_dim else None
        self.global_mlp = None
        if global_dim:
            self.global_mlp = nn.Sequential(
                nn.Linear(global_dim, hidden_dims[-1]),
                nn.ReLU(),
                nn.Dropout(dropout)
            )

        self.convs = nn.ModuleList()
        self.conv_norms = nn.ModuleList()
        in_dim = node_dim
        for h_dim in hidden_dims:
            self.convs.append(GATConv(
                in_channels=in_dim,
                out_channels=h_dim // gat_heads,
                heads=gat_heads,
                concat=True,
                dropout=dropout,
                edge_dim=edge_dim if edge_dim else None,
                add_self_loops=True
            ))
            self.conv_norms.append(nn.LayerNorm(h_dim))
            in_dim = h_dim

        self.dropout = nn.Dropout(dropout)
        self.final_dim = hidden_dims[-1] * (2 if global_dim else 1)
        self.output_mlp = nn.Sequential(
            nn.Linear(self.final_dim, self.final_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(self.final_dim // 2, 1)
        )

    def forward(self, data, return_feat=False):
        x = self.node_norm(data.x)
        edge_attr = getattr(data, 'edge_attr', None)
        if self.edge_norm is not None and edge_attr is not None:
            edge_attr = self.edge_norm(edge_attr)

        u = getattr(data, 'u', None)
        if self.global_norm is not None and u is not None:
            u = self.global_norm(u)

        for conv, conv_norm in zip(self.convs, self.conv_norms):
            x = conv(x, data.edge_index, edge_attr=edge_attr)
            x = conv_norm(x)
            x = F.elu(x)
            x = self.dropout(x)

        node_pool = global_mean_pool(x, data.batch)
        h = torch.cat([node_pool, self.global_mlp(u)], dim=1) if u is not None else node_pool
        out = self.output_mlp(h).squeeze(-1)
        return (out, h) if return_feat else out

class GateNet(nn.Module):
    """Learn sample-wise weights over the five teachers."""
    def __init__(self, in_dim, hidden_dim, num_teachers):
        super().__init__()
        self.fc1 = nn.Linear(in_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, num_teachers)
    def forward(self, h):
        a = F.relu(self.fc1(h))
        return F.softmax(self.fc2(a), dim=-1)

class Adapter(nn.Module):
    """Project student features to the teacher feature dimension."""
    def __init__(self, dim_s, dim_t):
        super().__init__()
        self.linear = nn.Linear(dim_s, dim_t)
    def forward(self, h):
        return self.linear(h)

def create_data_loader(graph_list, batch_size=32, shuffle=True):
    """Convert stored graph dictionaries to PyG Data objects."""
    data_list = []
    for g in graph_list:
        data_list.append(Data(
            x=g['x'], edge_index=g['edge_index'],
            edge_attr=g.get('edge_attr', None), u=g.get('u', None),
            y=g['y'], y_soft=g.get('y_soft', None)
        ))
    return DataLoader(data_list, batch_size=batch_size, shuffle=shuffle)

def train_model(
    train_dir, val_dir, teacher_paths, save_path,
    gate_hidden=128, hint_lambda=5.0, weight_ratio=(0.6,0.4),
    hidden_dims=[128,128], dropout=0.1, gat_heads=4,
    epochs=500, batch_size=64, lr=1e-3, min_lr=1e-4,
    lr_patience=20, es_patience=50,
    seed=42
):
    """Train one student configuration and return its best validation R²."""
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    set_seed(seed)
    
    train_g = torch.load(os.path.join(train_dir, 'graph_data.pt'))
    val_g = torch.load(os.path.join(val_dir, 'graph_data.pt'))

    ys = torch.stack([g['y'] for g in train_g]).view(-1)
    y_mean, y_std = ys.mean().item(), ys.std().item()+1e-8
    for g in train_g:
        g['y'] = (g['y'] - y_mean) / y_std
        g.setdefault('y_soft', g['y'])
    for g in val_g:
        g['y'] = (g['y'] - y_mean) / y_std
        g.setdefault('y_soft', g['y'])

    tr_loader = create_data_loader(train_g, batch_size, True)
    va_loader = create_data_loader(val_g, batch_size, False)
    
    sample = train_g[0]
    n_dim = sample['x'].size(1)
    e_dim = sample.get('edge_attr', None).size(1) if sample.get('edge_attr') is not None else 0
    g_dim = sample.get('u', None).size(1) if sample.get('u') is not None else 0
    student = EnhancedGAT(n_dim, e_dim, g_dim, hidden_dims, dropout, gat_heads).to(device)
    
    teachers = []
    for p in teacher_paths:
        ck = torch.load(p, map_location=device)
        if ck.get('model_type') != 'gat':
            raise ValueError(f"Expected a GAT teacher checkpoint: {p}")
        teacher_heads = ck.get('gat_heads', gat_heads)
        t = GATTeacher(
            ck['node_dim'], ck.get('edge_dim', 0), ck.get('global_dim', 0),
            ck['hidden_dims'], ck['dropout'], teacher_heads
        ).to(device)
        t.load_state_dict(ck['model_state_dict'], strict=True)
        if t.final_dim != student.final_dim:
            raise ValueError(
                f"Teacher/student feature dimensions differ: teacher={t.final_dim}, student={student.final_dim}"
            )
        t.eval()
        for parameter in t.parameters():
            parameter.requires_grad_(False)
        teachers.append(t)
    print(f"Loaded {len(teachers)} teachers.")

    # Gate & Adapter
    K = len(teachers)
    gate = GateNet(student.final_dim, gate_hidden, K).to(device)
    adapter = Adapter(student.final_dim, student.final_dim).to(device)
    optimizer = optim.Adam(list(student.parameters()) + list(gate.parameters()) + list(adapter.parameters()), lr=lr, weight_decay=1e-5)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', factor=0.5, patience=lr_patience, min_lr=min_lr)
    scaler = GradScaler(enabled=(device.type=='cuda'))
    best_r2, patience = -1e9, 0
    history = {'loss': [], 'train_r2': [], 'val_r2': []}

    def eval_loader(loader):
        student.eval()
        ys, ps = [], []
        with torch.no_grad():
            for b in loader:
                b = b.to(device)
                out, _ = student(b, return_feat=True)
                ys.append(b.y.view(-1).cpu().numpy())
                ps.append(out.cpu().numpy())
        return r2_score(np.concatenate(ys), np.concatenate(ps))

    for epoch in range(1, epochs + 1):
        student.train()
        total_loss = 0
        for batch in tr_loader:
            batch = batch.to(device)
            with autocast(enabled=(device.type == 'cuda')):
                pred_s, h_s = student(batch, return_feat=True)
                Ht = torch.stack([t(batch, return_feat=True)[1] for t in teachers], dim=1)
                w = gate(h_s)
                Ht_g = (w.unsqueeze(-1) * Ht).sum(dim=1)
                loss_hint = F.mse_loss(adapter(h_s), Ht_g)
                if batch.y_soft.numel() != w.size(0) * K:
                    raise ValueError(
                        f"Unexpected y_soft size: got={batch.y_soft.numel()}, expected={w.size(0) * K}"
                    )
                soft_targets = batch.y_soft.reshape(w.size(0), K)
                fused = (w * soft_targets).sum(dim=1)
                pred_f = weight_ratio[0] * pred_s + weight_ratio[1] * fused
                loss = (weight_ratio[0] * F.mse_loss(pred_f, batch.y.view(-1)) +
                        weight_ratio[1] * F.mse_loss(pred_s, fused) +
                        hint_lambda * loss_hint)
            optimizer.zero_grad()
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            scaler.step(optimizer)
            scaler.update()
            total_loss += loss.item()

        
        train_r2 = eval_loader(tr_loader)
        val_r2 = eval_loader(va_loader)
        avg_loss = total_loss / len(tr_loader)
        lr_now = optimizer.param_groups[0]['lr']
        history['loss'].append(avg_loss)
        history['train_r2'].append(train_r2)
        history['val_r2'].append(val_r2)

        
        if epoch % 30 == 0:
            print(f"Epoch {epoch} | Loss {avg_loss:.4f} | Train R2 {train_r2:.4f} | Val R2 {val_r2:.4f} | LR {lr_now:.1e}")

        scheduler.step(avg_loss)

        if val_r2 > best_r2:
            best_r2, patience = val_r2, 0
            torch.save({
                'model_state_dict': student.state_dict(),
                'gate_state': gate.state_dict(),
                'adapter_state': adapter.state_dict(),
                'y_mean': y_mean,
                'y_std': y_std,
                'history': history,
                'node_dim': n_dim,
                'edge_dim': e_dim,
                'global_dim': g_dim,
                'hidden_dims': hidden_dims,
                'dropout': dropout,
                'gat_heads': gat_heads,
                'model_type': 'gat'
            }, save_path)
            if epoch % 30 != 0:
                print(f"Saved best model at Epoch {epoch}")
        else:
            patience += 1
            if patience >= es_patience:
                print(f"Early stopping triggered at epoch {epoch}")
                break

    print(f"Training complete, best Val R2={best_r2:.4f}")

    return best_r2, save_path

if __name__ == "__main__":
    seeds = [0, 8, 42, 100, 456, 618, 1189, 2025, 2077, 2048]
    hint_lambdas = [1, 5, 10, 20]
    weight_ratios = [(0.4, 0.6), (0.5, 0.5), (0.6, 0.4), (0.7, 0.3)]
    gate_hiddens = [64, 128, 256]

    # Set paths before running. Teacher order must match y_soft:
    # qcut, elem, molwt, fingerprint, scaffold.
    train_dir = None
    val_dir = None
    teacher_paths = []
    save_root = None
    if not train_dir or not val_dir or not save_root:
        raise ValueError("Set train_dir, val_dir, and save_root before running.")
    if len(teacher_paths) != 5:
        raise ValueError("Provide five teacher checkpoints in the required order.")

    os.makedirs(save_root, exist_ok=True)
    epochs, batch_size = 1000, 64
    lr, min_lr = 1e-3, 5e-5
    lr_patience, es_patience = 30, 100
    hidden_dims, dropout = [128, 128], 0.1

    results, grid_rows = [], []
    for hint_lambda in hint_lambdas:
        for weight_ratio in weight_ratios:
            for gate_hidden in gate_hiddens:
                combo_key = (
                    f"hl{hint_lambda}_wr{weight_ratio[0]}_{weight_ratio[1]}_"
                    f"gh{gate_hidden}"
                )
                validation_scores = []
                for seed in seeds:
                    set_seed(seed)
                    checkpoint_path = os.path.join(
                        save_root, f"student_{combo_key}_seed{seed}.pt"
                    )
                    best_val_r2, _ = train_model(
                        train_dir=train_dir,
                        val_dir=val_dir,
                        teacher_paths=teacher_paths,
                        save_path=checkpoint_path,
                        gate_hidden=gate_hidden,
                        hint_lambda=hint_lambda,
                        weight_ratio=weight_ratio,
                        hidden_dims=hidden_dims,
                        dropout=dropout,
                        gat_heads=4,
                        epochs=epochs,
                        batch_size=batch_size,
                        lr=lr,
                        min_lr=min_lr,
                        lr_patience=lr_patience,
                        es_patience=es_patience,
                        seed=seed,
                    )
                    validation_scores.append(best_val_r2)
                    grid_rows.append({
                        "config": combo_key,
                        "seed": seed,
                        "best_validation_r2": best_val_r2,
                        "checkpoint": os.path.basename(checkpoint_path),
                    })

                results.append({
                    "combo": combo_key,
                    "r2_list": validation_scores,
                    "r2_mean": float(np.mean(validation_scores)),
                    "r2_std": float(np.std(validation_scores)),
                })

    pd.DataFrame(grid_rows).to_csv(
        os.path.join(save_root, "grid_search_validation.csv"), index=False
    )
    with open(os.path.join(save_root, "results_summary.pkl"), "wb") as file:
        pickle.dump(results, file)

    print("GAT hyperparameter search complete.")
